In [ ]:
# ============================================
# EXPERIMENT 4
# Comparative Study of Deep CNN Architectures
# Using Transfer Learning
# ============================================

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("TensorFlow version:", tf.__version__)

# Check GPU
print("\nGPU:")
print(tf.config.list_physical_devices("GPU"))

# Create folder for EPS figures
os.makedirs("eps_figures", exist_ok=True)

# ============================================
# PLOT FORMAT
# ============================================

plt.rcParams.update({
    "font.size": 16,
    "font.weight": "bold",
    "axes.labelsize": 16,
    "axes.labelweight": "bold",
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16
})

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# ============================================
# LOAD CIFAR-10
# ============================================

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)

print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)

# Normalize pixel values to [0,1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("\nNormalized pixel range:")
print("Minimum:", x_train.min())
print("Maximum:", x_train.max())

In [ ]:
# ============================================
# CLASS NAMES
# ============================================

class_names = [
    "Airplane",
    "Automobile",
    "Bird",
    "Cat",
    "Deer",
    "Dog",
    "Frog",
    "Horse",
    "Ship",
    "Truck"
]

NUM_CLASSES = 10

In [ ]:
# ============================================
# DISPLAY 10 SAMPLE IMAGES
# ============================================

plt.figure(figsize=(16, 8))

for i in range(10):
    plt.subplot(2, 5, i + 1)

    plt.imshow(x_train[i])

    plt.title(
        class_names[y_train[i][0]],
        fontsize=16,
        fontweight="bold"
    )

    plt.axis("off")

plt.tight_layout()

plt.savefig(
    "eps_figures/01_cifar10_samples.eps",
    format="eps",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================
# RESIZE CIFAR-10 IMAGES
# ============================================

IMG_SIZE = 96

# Resize datasets
x_train_resized = tf.image.resize(x_train, (IMG_SIZE, IMG_SIZE))
x_test_resized = tf.image.resize(x_test, (IMG_SIZE, IMG_SIZE))

print("Original image shape:", x_train.shape)
print("Resized image shape:", x_train_resized.shape)

In [ ]:
# ============================================
# ONE-HOT ENCODING
# ============================================

NUM_CLASSES = 10

y_train_cat = keras.utils.to_categorical(
    y_train,
    NUM_CLASSES
)

y_test_cat = keras.utils.to_categorical(
    y_test,
    NUM_CLASSES
)

print("Label shape:", y_train_cat.shape)

In [ ]:
# ============================================
# VGG16 TRANSFER LEARNING MODEL
# ============================================

from tensorflow.keras.applications import VGG16

base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze convolutional base
base_model.trainable = False

# Add new classifier
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(
    256,
    activation="relu"
)(x)

x = layers.Dropout(0.5)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = Model(inputs, outputs)

model.summary()

In [ ]:
# ============================================
# COMPILE MODEL
# ============================================

optimizer = keras.optimizers.Adam(
    learning_rate=0.001
)

model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

In [ ]:
# ============================================
# TRAINING
# ============================================

EPOCHS = 10
BATCH_SIZE = 32

start_time = time.time()

history = model.fit(
    x_train_resized,
    y_train_cat,

    validation_split=0.1,

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,

    verbose=1
)

training_time = time.time() - start_time

print(f"\nTraining Time: {training_time:.2f} seconds")

In [ ]:
# ============================================
# TRAINING + VALIDATION ACCURACY
# ============================================

plt.figure(figsize=(10, 7))

plt.plot(
    history.history["accuracy"],
    linewidth=2,
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    linewidth=2,
    label="Validation Accuracy"
)

plt.xlabel("Epoch", fontweight="bold", fontsize=16)
plt.ylabel("Accuracy", fontweight="bold", fontsize=16)
plt.title("Training and Validation Accuracy",
          fontweight="bold",
          fontsize=16)

plt.legend()
plt.grid(True, alpha=0.3)

plt.savefig(
    "eps_figures/02_training_validation_accuracy.eps",
    format="eps",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================
# TRAINING + VALIDATION LOSS
# ============================================

plt.figure(figsize=(10, 7))

plt.plot(
    history.history["loss"],
    linewidth=2,
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    linewidth=2,
    label="Validation Loss"
)

plt.xlabel("Epoch", fontweight="bold", fontsize=16)
plt.ylabel("Loss", fontweight="bold", fontsize=16)
plt.title("Training and Validation Loss",
          fontweight="bold",
          fontsize=16)

plt.legend()
plt.grid(True, alpha=0.3)

plt.savefig(
    "eps_figures/03_training_validation_loss.eps",
    format="eps",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================
# EVALUATION BEFORE FINE-TUNING
# ============================================

test_loss, test_accuracy = model.evaluate(
    x_test_resized,
    y_test_cat,
    batch_size=BATCH_SIZE,
    verbose=1
)

print("\nBefore Fine-Tuning")
print("-------------------")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")

In [ ]:
# ============================================
# FINE-TUNING VGG16
# ============================================

# First keep everything frozen
base_model.trainable = True

# Freeze all layers
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze the final convolutional block
for layer in base_model.layers:
    if layer.name.startswith("block5"):
        layer.trainable = True

# Display trainable layers
print("Trainable layers:")
for layer in base_model.layers:
    if layer.trainable:
        print(layer.name)

In [ ]:
# ============================================
# COMPILE AND FINE-TUNE VGG16
# ============================================

# Lower learning rate for fine-tuning to preserve pre-trained weights
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n--- Fine-Tuning VGG16 ---")
FINE_TUNE_EPOCHS = 5

start_time = time.time()
history_vgg_ft = model.fit(
    x_train_resized,
    y_train_cat,
    validation_split=0.1,
    epochs=FINE_TUNE_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)
vgg_ft_time = time.time() - start_time

# Evaluate Fine-Tuned VGG16
vgg_test_loss, vgg_test_acc = model.evaluate(x_test_resized, y_test_cat, verbose=0)
vgg_preds_prob = model.predict(x_test_resized, batch_size=BATCH_SIZE)
vgg_preds = np.argmax(vgg_preds_prob, axis=1)

print(f"\nVGG16 Fine-Tuned Test Accuracy: {vgg_test_acc * 100:.2f}%")

# Plot VGG16 Fine-Tuning Accuracy & Loss Curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_vgg_ft.history["accuracy"], label="Fine-Tune Train Acc", linewidth=2)
plt.plot(history_vgg_ft.history["val_accuracy"], label="Fine-Tune Val Acc", linewidth=2)
plt.xlabel("Epoch", fontweight="bold")
plt.ylabel("Accuracy", fontweight="bold")
plt.title("VGG16 Fine-Tuning Accuracy", fontweight="bold")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_vgg_ft.history["loss"], label="Fine-Tune Train Loss", linewidth=2)
plt.plot(history_vgg_ft.history["val_loss"], label="Fine-Tune Val Loss", linewidth=2)
plt.xlabel("Epoch", fontweight="bold")
plt.ylabel("Loss", fontweight="bold")
plt.title("VGG16 Fine-Tuning Loss", fontweight="bold")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("eps_figures/04_vgg16_finetuning_curves.eps", format="eps", bbox_inches="tight")
plt.show()

# ============================================
# COMPARATIVE ARCHITECTURE: RESNET50 MODEL
# ============================================

from tensorflow.keras.applications import ResNet50

print("\n--- Building ResNet50 Transfer Learning Model ---")

resnet_base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

resnet_base.trainable = False  # Freeze base layers initially

inputs_rn = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x_rn = resnet_base(inputs_rn, training=False)
x_rn = layers.GlobalAveragePooling2D()(x_rn)
x_rn = layers.Dense(256, activation="relu")(x_rn)
x_rn = layers.Dropout(0.5)(x_rn)
outputs_rn = layers.Dense(NUM_CLASSES, activation="softmax")(x_rn)

resnet_model = Model(inputs_rn, outputs_rn)

resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n--- Training ResNet50 Feature Extractor ---")
start_time = time.time()
history_resnet = resnet_model.fit(
    x_train_resized,
    y_train_cat,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)
resnet_train_time = time.time() - start_time

resnet_test_loss, resnet_test_acc = resnet_model.evaluate(x_test_resized, y_test_cat, verbose=0)
resnet_preds_prob = resnet_model.predict(x_test_resized, batch_size=BATCH_SIZE)
resnet_preds = np.argmax(resnet_preds_prob, axis=1)

print(f"\nResNet50 Test Accuracy: {resnet_test_acc * 100:.2f}%")

# ============================================
# COMPARATIVE STUDY PLOTS & METRICS
# ============================================

# 1. Accuracy vs Loss Comparison Plot across Architectures
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(history.history["val_accuracy"], label="VGG16 Feature Extractor", linewidth=2)
plt.plot(history_resnet.history["val_accuracy"], label="ResNet50 Feature Extractor", linewidth=2)
plt.xlabel("Epoch", fontweight="bold")
plt.ylabel("Validation Accuracy", fontweight="bold")
plt.title("Architecture Accuracy Comparison", fontweight="bold")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history["val_loss"], label="VGG16 Feature Extractor", linewidth=2)
plt.plot(history_resnet.history["val_loss"], label="ResNet50 Feature Extractor", linewidth=2)
plt.xlabel("Epoch", fontweight="bold")
plt.ylabel("Validation Loss", fontweight="bold")
plt.title("Architecture Loss Comparison", fontweight="bold")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("eps_figures/05_architecture_accuracy_loss_comparison.eps", format="eps", bbox_inches="tight")
plt.show()

# 2. Confusion Matrices Comparison
cm_vgg = confusion_matrix(y_test.flatten(), vgg_preds)
cm_resnet = confusion_matrix(y_test.flatten(), resnet_preds)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# VGG16 Heatmap
im0 = axes[0].imshow(cm_vgg, interpolation="nearest", cmap=plt.cm.Blues)
axes[0].set_title("VGG16 Confusion Matrix", fontweight="bold")
fig.colorbar(im0, ax=axes[0])
tick_marks = np.arange(NUM_CLASSES)
axes[0].set_xticks(tick_marks)
axes[0].set_xticklabels(class_names, rotation=45, ha="right")
axes[0].set_yticks(tick_marks)
axes[0].set_yticklabels(class_names)
axes[0].set_ylabel("True Label", fontweight="bold")
axes[0].set_xlabel("Predicted Label", fontweight="bold")

# ResNet50 Heatmap
im1 = axes[1].imshow(cm_resnet, interpolation="nearest", cmap=plt.cm.Greens)
axes[1].set_title("ResNet50 Confusion Matrix", fontweight="bold")
fig.colorbar(im1, ax=axes[1])
axes[1].set_xticks(tick_marks)
axes[1].set_xticklabels(class_names, rotation=45, ha="right")
axes[1].set_yticks(tick_marks)
axes[1].set_yticklabels(class_names)
axes[1].set_ylabel("True Label", fontweight="bold")
axes[1].set_xlabel("Predicted Label", fontweight="bold")

plt.tight_layout()
plt.savefig("eps_figures/06_confusion_matrices_comparison.eps", format="eps", bbox_inches="tight")
plt.show()

# 3. Overall Performance Comparison Metrics Table & Chart
y_true = y_test.flatten()

metrics_vgg = {
    "Accuracy": accuracy_score(y_true, vgg_preds),
    "Precision": precision_score(y_true, vgg_preds, average="macro"),
    "Recall": recall_score(y_true, vgg_preds, average="macro"),
    "F1-Score": f1_score(y_true, vgg_preds, average="macro"),
    "Training Time (s)": training_time + vgg_ft_time
}

metrics_resnet = {
    "Accuracy": accuracy_score(y_true, resnet_preds),
    "Precision": precision_score(y_true, resnet_preds, average="macro"),
    "Recall": recall_score(y_true, resnet_preds, average="macro"),
    "F1-Score": f1_score(y_true, resnet_preds, average="macro"),
    "Training Time (s)": resnet_train_time
}

comparison_df = pd.DataFrame([metrics_vgg, metrics_resnet], index=["VGG16 (Fine-Tuned)", "ResNet50"])
print("\n=======================================================")
print("  EXPERIMENT 4: COMPARATIVE ARCHITECTURE SUMMARY TABLE ")
print("=======================================================")
print(comparison_df.round(4))

# 4. Comparative Bar Chart
fig, ax1 = plt.subplots(figsize=(10, 6))

x_indices = np.arange(len(comparison_df.index))
bar_width = 0.35

rects1 = ax1.bar(x_indices - bar_width/2, comparison_df["Accuracy"] * 100, bar_width, label="Accuracy (%)", color="navy", alpha=0.8)

ax2 = ax1.twinx()
rects2 = ax2.bar(x_indices + bar_width/2, comparison_df["Training Time (s)"], bar_width, label="Training Time (s)", color="darkred", alpha=0.8)

ax1.set_xlabel("Architecture", fontweight="bold")
ax1.set_ylabel("Accuracy (%)", fontweight="bold", color="navy")
ax2.set_ylabel("Training Time (Seconds)", fontweight="bold", color="darkred")
ax1.set_xticks(x_indices)
ax1.set_xticklabels(comparison_df.index, fontweight="bold")
plt.title("Comparative Performance: Accuracy vs Training Speed", fontweight="bold")

plt.tight_layout()
plt.savefig("eps_figures/07_performance_time_comparison.eps", format="eps", bbox_inches="tight")
plt.show()

print("\nClassification Report (VGG16):")
print(classification_report(y_true, vgg_preds, target_names=class_names))

print("\nClassification Report (ResNet50):")
print(classification_report(y_true, resnet_preds, target_names=class_names))